In [ ]:
import numpy as np
import pandas as pd
from datetime import timedelta
import matplotlib.pyplot as plt
import xarray

In [ ]:
"""
GENERATE HOURLY TIMESTEP SERIES (UTC)
"""

timestamps = pd.date_range(
    start="2010-01-01 00:00:00",
    end="2045-12-31 23:00:00",
    freq="h",
    tz="UTC"
)

df_cal = pd.DataFrame({"timestamp_utc": timestamps})
print(f"Rows: {len(df_cal):,}")
print(f"First: {df_cal['timestamp_utc'].iloc[0]}")
print(f"Last: {df_cal['timestamp_utc'].iloc[-1]}")

In [ ]:
"""
CALENDAR FIELDS
"""

df_cal["year"] = df_cal["timestamp_utc"].dt.year
df_cal["month"] = df_cal["timestamp_utc"].dt.month
df_cal["day"] = df_cal["timestamp_utc"].dt.day
df_cal["hour"] = df_cal["timestamp_utc"].dt.hour
df_cal["day_of_week"] = df_cal["timestamp_utc"].dt.day_of_week  # 0=Mon, 6=Sun
df_cal["is_weekend"] = df_cal["day_of_week"].isin([5, 6]).astype(int)

print(df_cal.head(25))
print(df_cal.dtypes)

In [ ]:
"""
LOAD AND JOIN HOLIDAY FLAGS
"""

holidays = pd.read_csv("holiday_calendar_2010_2045.csv", parse_dates=["date"])

# Strip timezone from _date so it matches the timezone-naive holiday CSV dates
df_cal["_date"] = df_cal["timestamp_utc"].dt.normalize().dt.tz_localize(None)

df_cal = df_cal.merge(
    holidays[["date", "is_holiday_eng_wales", "is_holiday_scotland", "is_holiday_gb_any"]],
    left_on="_date",
    right_on="date",
    how="left"
).drop(columns=["_date", "date"])

print(df_cal.head())
print(df_cal.dtypes)

In [ ]:
print("=== Quality checks ===")

# No missing hours
expected = pd.date_range("2010-01-01", "2045-12-31 23:00", freq="h", tz="UTC")
missing_hours = expected.difference(df_cal["timestamp_utc"])
print(f"Missing hours: {len(missing_hours)} {'No missing hours.' if len(missing_hours)==0 else '!'}")

# No duplicates
dupes = df_cal["timestamp_utc"].duplicated().sum()
print(f"Duplicate timestamps: {dupes} {'No missing hours.' if dupes==0 else '!'}")

# Correct start and end
print(f"Start: {df_cal['timestamp_utc'].iloc[0]}  {'No missing hours.' if str(df_cal['timestamp_utc'].iloc[0])=='2010-01-01 00:00:00+00:00' else '!'}")
print(f"End: {df_cal['timestamp_utc'].iloc[-1]}  {'No missing hours.' if str(df_cal['timestamp_utc'].iloc[-1])=='2045-12-31 23:00:00+00:00' else '!'}")

# Holiday flags fully populated (no NaNs from the join)
for col in ["is_holiday_eng_wales", "is_holiday_scotland", "is_holiday_gb_any", "is_weekend"]:
    n = df_cal[col].isna().sum()
    print(f"{col} NaNs: {n} {'No missing hours.' if n==0 else '!'}")

# Sense checks on flag values
print(f"\nWeekend hours: {df_cal['is_weekend'].sum():,}  (expect ~90,144)")
print(f"E&W holiday hours: {df_cal['is_holiday_eng_wales'].sum():,}")
print(f"Scotland holiday hours: {df_cal['is_holiday_scotland'].sum():,}")
print(f"GB any holiday hours: {df_cal['is_holiday_gb_any'].sum():,}")

print(f"\nColumns: {list(df_cal.columns)}")
print(df_cal.dtypes)

In [ ]:
df_cal.to_parquet("calendar_hourly_2010_2045.parquet", index=False)
print("Saved: calendar_hourly_2010_2045.parquet")

# Verify by reloading
check = pd.read_parquet("calendar_hourly_2010_2045.parquet")
print(f"Shape: {check.shape}") # expect (315576, 10)
print(check.head())